# M5: seleção de parâmetros e treino final (Colab, GPU)
Antes: *Runtime > Alterar tipo de ambiente de execução > GPU* e criar o Secret `GITHUB_TOKEN` (ícone de chave, com acesso ao notebook ligado). Faça o push dos commits antes: o notebook usa o que estiver no GitHub.

Arquivos locais necessários (upload na célula 3): `data/processed/m5_store_dept.csv` e `data/processed/pred_m5_arima.npy` (previsões do ARIMA, usadas só na figura).

In [ ]:
# Clona o repo privado com o token dos Secrets (o token não é gravado no notebook nem em .git/config)
import base64, os, subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
if not os.path.exists('/content/mtst-demanda-varejo'):
    subprocess.run(['git', '-c', f'http.extraheader=AUTHORIZATION: basic {auth}', 'clone',
                    'https://github.com/VanthuirMaia/mtst-demanda-varejo.git',
                    '/content/mtst-demanda-varejo'], check=True)
%cd /content/mtst-demanda-varejo
!git log --oneline -3

In [ ]:
# torch, pandas e numpy já vêm no Colab; falta só isto (statsforecast não é necessário aqui)
!pip install -q einops yacs

In [ ]:
# Upload dos dados processados (não versionados): m5_store_dept.csv e pred_m5_arima.npy
import os, shutil
from google.colab import files
os.makedirs('data/processed', exist_ok=True)
if not (os.path.exists('data/processed/m5_store_dept.csv') and os.path.exists('data/processed/pred_m5_arima.npy')):
    for nome in files.upload():
        shutil.move(nome, f'data/processed/{nome}')
!ls -la data/processed

In [ ]:
# Confere GPU e que o M5 carrega só com o CSV processado (sem os CSVs brutos)
import torch
assert torch.cuda.is_available(), 'Sem GPU: ative em Ambiente de execução > Alterar tipo'
print(torch.cuda.get_device_name(0))
from src.data import load_data
from src import config as C
m = load_data(); C.imprimir_splits()
assert m.shape == (1941, 70)

In [ ]:
# (Opcional) retomada após desconexão: guarda/restaura m5_*.csv no Google Drive
from google.colab import drive
drive.mount('/content/drive')
BK = '/content/drive/MyDrive/mtst_m5_backup'
os.makedirs(BK, exist_ok=True)
import glob
for f in glob.glob(f'{BK}/m5_*'):            # restaura o que já foi feito
    shutil.copy(f, 'results/tables/')
def salvar_backup():
    for f in glob.glob('results/tables/m5_*') + glob.glob('results/figures/m5_*'):
        shutil.copy(f, BK)
    for f in glob.glob('data/processed/pred_m5_*'):
        shutil.copy(f, BK)
print(os.listdir(BK))

## 1. Seleção de parâmetros (só treino e validação)
Grade: MTST 8 configs, PatchTST 12 configs (20 no total), 15 épocas cada. Retomável: se desconectar, rode de novo (pula o que já está em `m5_selecao.csv`).

In [ ]:
!python -u -m src.selecao --backup {BK}
salvar_backup()

In [ ]:
import pandas as pd
pd.read_csv('results/tables/m5_selecao.csv').sort_values(['modelo', 'val_mse_norm'])

## 2. Treino final (melhor config de cada modelo, seeds 42, 43, 44) e avaliação no teste

In [ ]:
# Retomada de previsões do ARIMA/Transformers (seed 42) a partir do backup, se existirem
for f in glob.glob(f'{BK}/pred_m5_*'):
    if not os.path.exists(f'data/processed/{os.path.basename(f)}'):
        shutil.copy(f, 'data/processed/')
!python -u -m src.final --backup {BK}
salvar_backup()

In [ ]:
pd.read_csv('results/tables/m5_comparacao.csv')

In [ ]:
# Compacta tabelas e figuras do M5 para download
!zip -q -r m5_resultados.zip results/tables/m5_* results/figures/m5_*
from google.colab import files
files.download('m5_resultados.zip')